In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 Extreme Logistic Regressor with Undersampling First & Step-Wise Class Weighting (`models/lr_feng_esi1_extreme.ipynb`)

This notebook trains a **Binary Logistic Regressor** for **ESI 1 vs Not ESI 1** with **Majority Class Undersampling Applied First** and **Direct Step-Wise Class Weight Optimization**:

### System Architecture & Workflow
1. **Majority Class Undersampling First**: Applies 1:1 balanced undersampling of majority class (`"not_1"`) relative to ESI 1 rows before model training.
2. **Feature Set (13 Clinical Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and 10 vital sign anomaly flags.
3. **Step-Wise Class Weight Optimization**: Iterates directly through numeric class weight steps ($w \in [0.5, 10.0]$) to evaluate Precision and Recall trajectories.
4. **Diagnostic Plotting**: Exports trajectory plot to `plots/lr_feng_esi1_weight_tuning.png` (Precision & Recall vs Class Weight Step).
5. **Reports & Model Export**: Evaluates optimal model, prints Actual vs Predicted class counts, Precision, Recall, and PR-AUC, and exports CSV reports (`reports/lr_feng_esi1_tuning_results.csv`, `reports/lr_feng_esi1_val_report.csv`, `reports/lr_feng_esi1_test_report.csv`). Artifact saved to `deploy/lr_feng_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 13 FE Inputs, & Apply Majority Class Undersampling FIRST
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 13 Clinical Feature Engineering flags
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

raw_esi <- as.character(raw_df[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))

initial_rows <- nrow(df_feng)
df_feng <- na.omit(df_feng)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining: %d)\n", initial_rows - nrow(df_feng), nrow(df_feng)))
# ---------------------------------------------------------
# APPLY MAJORITY CLASS UNDERSAMPLING FIRST (1:1 Balanced Ratio)
# ---------------------------------------------------------
idx_1     <- which(df_feng$target_layer1 == "1")
idx_not_1 <- which(df_feng$target_layer1 == "not_1")
n_not_1_keep <- length(idx_1)  # 1:1 ratio matching ESI 1 count
kept_not_1   <- sample(idx_not_1, size = n_not_1_keep)
kept_1       <- idx_1
df_feng <- df_feng[sort(c(kept_1, kept_not_1)), ]
cat(sprintf("Undersampling Applied First: %d ESI 1 rows & %d undersampled 'not_1' rows -> %d total rows x %d cols\n",
            length(kept_1), length(kept_not_1), nrow(df_feng), ncol(df_feng)))
cat("Undersampled Binary Target Distribution:\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Iterative Class Weight Stepping Optimization (Direct Numeric Weight Steps)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Define Direct Numeric Weight Steps
class_weight_steps <- c(0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0)
feat_names <- setdiff(names(train_df), c("target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
results_list <- list()
cat(sprintf("Starting Step-Wise Class Weight Optimization over %d numeric weight steps...\n", length(class_weight_steps)))
for (iter in 1:length(class_weight_steps)) {
  w_step  <- class_weight_steps[iter]
  weights <- ifelse(train_df$target_layer1 == "1", w_step, 1.0)
  
  model <- multinom(formula_lr, data = train_df, weights = weights, trace = FALSE, MaxNWts = 5000)
  
  pred_val <- as.character(predict(model, newdata = val_df))
  pred_val[is.na(pred_val)] <- "not_1"
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(val_df$target_layer1, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  prob_res <- predict(model, newdata = val_df, type = "probs")
  prob_1   <- if (is.matrix(prob_res)) {
    if ("1" %in% colnames(prob_res)) prob_res[, "1"] else 1 - prob_res[, "not_1"]
  } else {
    1 - prob_res
  }
  pr_auc <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  
  results_list[[iter]] <- data.frame(
    Iteration    = iter,
    Class_Weight = w_step,
    Accuracy     = round(acc, 4),
    Precision    = round(prec, 4),
    Recall       = round(rec, 4),
    F1_Score     = round(f1, 4),
    PR_AUC       = round(pr_auc, 4)
  )
}
tuning_df <- do.call(rbind, results_list)
# Write Tuning Results to CSV
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
csv_tuning_path <- file.path(reports_dir, "lr_feng_esi1_tuning_results.csv")
write.csv(tuning_df, file = csv_tuning_path, row.names = FALSE)
cat("Tuning Results CSV written to:", csv_tuning_path, "\n\n")
cat("Step-Wise Class Weight Optimization complete! Top 5 configurations by Validation F1 Score:\n")
print(head(tuning_df[order(-tuning_df$F1_Score), ], 5))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Save Trajectory Diagnostic Plot (Precision & Recall vs Class Weight Step)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
df_plot <- tuning_df %>%
  select(Class_Weight, Precision, Recall, F1_Score, PR_AUC) %>%
  pivot_longer(cols = c("Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p1 <- ggplot(df_plot, aes(x = Class_Weight, y = Score, color = Metric)) +
  geom_line(size = 1.0) + geom_point(size = 2.0) +
  theme_minimal() +
  scale_color_manual(values = c("Precision" = "#d90429", "Recall" = "#ff4d6d", "F1_Score" = "#2b5c8f", "PR_AUC" = "#81b29a")) +
  labs(title = "Binary ESI 1: Precision, Recall & F1 Trajectories Across Class Weight Steps",
       subtitle = "Evaluating performance trajectory over varying direct class weight steps",
       x = "ESI 1 Class Weight Step Value", y = "Metric Value Score") +
  theme(plot.title = element_text(face = "bold", size = 12), legend.position = "top")
plot_path <- file.path(plots_dir, "lr_feng_esi1_weight_tuning.png")
ggsave(plot_path, plot = p1, width = 8.5, height = 4.5, dpi = 300)
cat("ESI 1 Step-Wise Weight Tuning Plot saved to:", plot_path, "\n")
p1

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Retrain Optimal Model & Export CSV Reports (Actual vs Predicted Count, Precision, Recall, PR-AUC)
# ---------------------------------------------------------
best_row <- tuning_df[which.max(tuning_df$F1_Score), ]
cat(sprintf("=== Selected Optimal Weight Configuration (Iteration %d) ===\n", best_row$Iteration))
cat(sprintf("  Class Weight Step : %.2f\n", best_row$Class_Weight))
cat(sprintf("  Validation F1     : %.4f\n", best_row$F1_Score))
cat(sprintf("  Validation Prec   : %.4f\n", best_row$Precision))
cat(sprintf("  Validation Rec    : %.4f\n\n", best_row$Recall))
final_weights <- ifelse(train_df$target_layer1 == "1", best_row$Class_Weight, 1.0)
lr_esi1_final <- multinom(formula_lr, data = train_df, weights = final_weights, trace = FALSE, MaxNWts = 5000)
evaluate_and_report_binary_esi1 <- function(model, data, set_name) {
  pred_val <- as.character(predict(model, newdata = data))
  pred_val[is.na(pred_val)] <- "not_1"
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(data$target_layer1, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  prob_res <- predict(model, newdata = data, type = "probs")
  prob_1   <- if (is.matrix(prob_res)) {
    if ("1" %in% colnames(prob_res)) prob_res[, "1"] else 1 - prob_res[, "not_1"]
  } else {
    1 - prob_res
  }
  pr_auc <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("1", "not_1"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 1 OPTIMAL LOGISTIC REGRESSOR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 1 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 1 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 1 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 1 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Class Count Comparison & Metrics Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(report_df)
}
# Generate Validation Report & Write to CSV
val_report <- evaluate_and_report_binary_esi1(lr_esi1_final, val_df, "Validation")
csv_val_path <- file.path(reports_dir, "lr_feng_esi1_val_report.csv")
write.csv(val_report, file = csv_val_path, row.names = FALSE)
cat("Validation CSV Report written to:", csv_val_path, "\n")
# Generate Test Report & Write to CSV
test_report <- evaluate_and_report_binary_esi1(lr_esi1_final, test_df, "Test")
csv_test_path <- file.path(reports_dir, "lr_feng_esi1_test_report.csv")
write.csv(test_report, file = csv_test_path, row.names = FALSE)
cat("Test CSV Report written to:", csv_test_path, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Optimal Binary ESI 1 Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
saveRDS(list(model = lr_esi1_final, preproc = preproc), file = model_path)
cat("Optimal Binary ESI 1 Logistic Regressor model saved to:", model_path, "\n")